# YOLO CIFAR10: Fine-Tuned Teacher → Pruned Student → Logits Distillation (Classification)

This notebook runs a YOLOv8n-cls **classification** pruning + logits distillation flow on CIFAR10:
1. Load a **CIFAR10-fine-tuned** YOLOv8n-cls teacher (10-class head).
2. Build a `MaseGraph` from the teacher weights and run unstructured pruning to create the student.
3. Distill teacher logits into the pruned student using CIFAR10 mini-batches (images + labels).
4. Evaluate teacher, pruned-only (no KD) student, and distilled student on CIFAR10 validation set.

In [1]:
TEACHER_CHECKPOINT = "data/cifar10_yolov8x_cls/runs/yolov8x_cls_cifar10_finetune/weights/best.pt"
STUDENT_CHECKPOINT = "data/cifar10_yolov8m_cls/runs/yolov8m_cls_cifar10_finetune/weights/best.pt"
DEVICE = "cuda"
DATA_ROOT = "./data"

IMAGE_SIZE = 32
BATCH_SIZE = 16

PRUNE_SPARSITY = 0.5
KD_ALPHA = 0.6
KD_TEMPERATURE = 4.0
EPOCHS = 3
lr = 5e-5

seed = 42

## Imports and setup

In [2]:
import copy
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo.yolov8 import MaseYoloClassificationModel, patch_yolo

from mase_kd.core.losses import DistillationLossConfig, hard_label_ce_loss, soft_logit_kl_loss
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")


/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


In [3]:
if DEVICE == "cuda" and not torch.cuda.is_available():
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")

torch.manual_seed(seed)
random.seed(seed)

Using device: cuda


## CIFAR10 image dataset and dataloaders

In [4]:
# Ultralytics trains YOLO on CIFAR10 with mean=(0,0,0), std=(1,1,1) 
cifar_transform_train = transforms.Compose([
    transforms.ToTensor(),
])
cifar_transform_eval = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True, transform=cifar_transform_train, download=True)
val_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False, transform=cifar_transform_eval, download=True)

# drop_last=True keeps batch size fixed for the FX-specialized pruned student graph.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

#print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

## Load CIFAR10-fine-tuned teacher (YOLOv8n-cls, 10 classes)

In [5]:

# Two-step loading is necessary for two reasons:
# 1. Ultralytics .pt files are pickled objects (not plain state dicts), so YOLO() is the
#    safe deserializer. ultra_teacher.model.yaml is the only clean way to read `nc` (and
#    other arch params) before we can instantiate MaseYoloClassificationModel(nc=nc).
# 2. Once YOLO() has reconstructed the live model, ultra_teacher.model.state_dict() gives
#    a standard PyTorch state dict that maps 1:1 to MaseYoloClassificationModel's weights.
#    We need MaseYoloClassificationModel (not ultra_teacher.model directly) so that the
#    teacher's forward returns a flat [batch, nc] tensor; the raw ultralytics
#    ClassificationModel returns a structured tuple in train mode, which would cause
#    _flatten_logits to concatenate it into [batch, 2*nc].

ultra_teacher = YOLO(TEACHER_CHECKPOINT)
nc = ultra_teacher.model.yaml.get("nc", 10)

teacher_cls_model = MaseYoloClassificationModel(cfg="yolov8x-cls.yaml", nc=nc)
teacher_cls_model = patch_yolo(teacher_cls_model)
teacher_cls_model.load_state_dict(ultra_teacher.model.state_dict())
teacher_cls_model = teacher_cls_model.to(DEVICE)
teacher_cls_model.eval()

print(f"Loaded CIFAR10-fine-tuned teacher: {TEACHER_CHECKPOINT}")
print(f"Teacher num_classes (nc): {nc}")


Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1      2320  ultralytics.nn.modules.conv.Conv             [3, 80, 3, 2]                 
  1                  -1  1    115520  ultralytics.nn.modules.conv.Conv             [80, 160, 3, 2]               
  2                  -1  3    436800  ultralytics.nn.modules.block.C2f             [160, 160, 3, True]           
  3                  -1  1    461440  ultralytics.nn.modules.conv.Conv             [160, 320, 3, 2]              
  4                  -1  6   3281920  ultralytics.nn.modules.block.C2f             [320, 320, 6, True]           
  5                  -1  1   1844480  ultralytics.nn.modules.conv.Conv             [320, 640, 3, 2]              
  6                  -1  6  13117440  ultralytics.nn.modules.block.C2f             [640, 640, 6, True]           
  7                  -1  1   7375360  ultralyt

## Build MASE graph from teacher weights and prune to create student

In [6]:

# Load the fine-tuned student checkpoint (yolov8n-cls) separately so the pruned
# student starts from its own weights rather than a copy of the larger teacher.
ultra_student = YOLO(STUDENT_CHECKPOINT)

student_seed_cls_model = MaseYoloClassificationModel(cfg="yolov8m-cls.yaml", nc=nc)
student_seed_cls_model = patch_yolo(student_seed_cls_model)
student_seed_cls_model.load_state_dict(ultra_student.model.state_dict())

# Initialize a MaseGraph
mg_cls = MaseGraph(student_seed_cls_model)
mg_cls, _ = passes.init_metadata_analysis_pass(mg_cls)

# Insert a dummy input for the common metadata analysis pass
trace_input_cls = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
placeholder_names_cls = [
    node.name for node in mg_cls.fx_graph.nodes if node.op == "placeholder"
]
dummy_in_cls = {name: trace_input_cls for name in placeholder_names_cls}

mg_cls, _ = passes.add_common_metadata_analysis_pass(
    mg_cls,
    pass_args={
        "dummy_in": dummy_in_cls,
    },
)

# Pruning config
pruning_config_cls = {
    "weight": {
        "sparsity": PRUNE_SPARSITY,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": PRUNE_SPARSITY,
        "method": "l1-norm",
        "scope": "local",
    },
}

# Apply pruning and extract the pruned student model.
mg_cls, _ = passes.prune_transform_pass(mg_cls, pass_args=pruning_config_cls)
student_cls_model = mg_cls.model.to(DEVICE)

# Snapshot the pruned student before KD training for baseline comparison.
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(DEVICE)
pruned_no_kd_model.eval()

print(f"Loaded CIFAR10-fine-tuned student seed: {STUDENT_CHECKPOINT}")
print(f"Pruning complete ({PRUNE_SPARSITY*100:.0f}% sparsity); pruned student ready.")


Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   2655744  ultralyt

INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_m_1_cv1_conv
INFO     Pruning module: model_2_m_1_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_m_2_cv1_conv
INFO     Pruning module: model_4_m_2_cv2_conv
INFO     Pruning module: model_4_m_3_cv1_conv
INFO     Pruning module: model_4_m_3_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: model_6_m_0_cv2

Loaded CIFAR10-fine-tuned student seed: data/cifar10_yolov8m_cls/runs/yolov8m_cls_cifar10_finetune/weights/best.pt
Pruning complete (50% sparsity); pruned student ready.


## Evaluate pruned model

In [7]:
import time

@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device):
    """Evaluate a model on the full validation loader. Returns top-1 accuracy, CE loss, and timing."""
    model.eval()
    batches = 0
    samples = 0
    total_forward_ms = 0.0
    correct_top1 = 0
    total_ce_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        batches += 1
        samples += images.shape[0]

        max_label = int(labels.max().item())
        if logits.shape[1] > max_label:
            preds = logits.argmax(dim=1)
            correct_top1 += int((preds == labels).sum().item())
            total_ce_loss += hard_label_ce_loss(logits, labels).item()

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "top1_acc": correct_top1 / max(samples, 1),
        "avg_ce_loss": total_ce_loss / max(batches, 1),
    }

pruned_no_kd_metrics = evaluate_model_on_cifar10_val(pruned_no_kd_model, val_loader, DEVICE)
print(f"Pruned student (no finetuning, no KD, {PRUNE_SPARSITY*100:.0f}% sparsity):")
print(
    f"  top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={pruned_no_kd_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={pruned_no_kd_metrics['samples']}"
)


Pruned student (no finetuning, no KD, 50% sparsity):
  top1_acc=57.47% | CE_loss=1.4056 | fwd_ms/batch=31.97 | samples=10000


## Finetune and evaluate pruned model (No Distillation)

In [8]:
# Fine-tune a copy of the pruned student with CE loss only (no distillation).
pruned_finetuned_model = copy.deepcopy(pruned_no_kd_model).to(DEVICE)
ft_optimizer = torch.optim.Adam(pruned_finetuned_model.parameters(), lr=lr)

for epoch in range(1, EPOCHS + 1):
    pruned_finetuned_model.train()
    total_loss = 0.0
    num_batches = len(train_loader)
    for batch_idx, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        ft_optimizer.zero_grad(set_to_none=True)

        outputs = pruned_finetuned_model(images)
        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        loss = hard_label_ce_loss(logits, labels)
        loss.backward()
        ft_optimizer.step()
        total_loss += loss.item()

        if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == num_batches:
            print(
                f"  Epoch {epoch}/{EPOCHS} | Batch {batch_idx:04d}/{num_batches} | "
                f"loss={loss.item():.6f}"
            )
    print(f"Epoch {epoch} avg loss: {total_loss / num_batches:.6f}")

pruned_finetuned_metrics = evaluate_model_on_cifar10_val(pruned_finetuned_model, val_loader, DEVICE)
print(f"\nPruned student (finetuned {EPOCHS} epochs, no KD, {PRUNE_SPARSITY*100:.0f}% sparsity):")
print(
    f"  top1_acc={pruned_finetuned_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={pruned_finetuned_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={pruned_finetuned_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={pruned_finetuned_metrics['samples']}"
)


  Epoch 1/3 | Batch 0001/3125 | loss=1.175313
  Epoch 1/3 | Batch 0100/3125 | loss=0.970657
  Epoch 1/3 | Batch 0200/3125 | loss=1.266231
  Epoch 1/3 | Batch 0300/3125 | loss=0.717836
  Epoch 1/3 | Batch 0400/3125 | loss=0.594920
  Epoch 1/3 | Batch 0500/3125 | loss=0.324089
  Epoch 1/3 | Batch 0600/3125 | loss=0.916608
  Epoch 1/3 | Batch 0700/3125 | loss=1.373882
  Epoch 1/3 | Batch 0800/3125 | loss=0.628070
  Epoch 1/3 | Batch 0900/3125 | loss=0.430892
  Epoch 1/3 | Batch 1000/3125 | loss=1.033243
  Epoch 1/3 | Batch 1100/3125 | loss=0.506934
  Epoch 1/3 | Batch 1200/3125 | loss=0.719073
  Epoch 1/3 | Batch 1300/3125 | loss=0.819343
  Epoch 1/3 | Batch 1400/3125 | loss=0.619671
  Epoch 1/3 | Batch 1500/3125 | loss=0.943168
  Epoch 1/3 | Batch 1600/3125 | loss=0.847735
  Epoch 1/3 | Batch 1700/3125 | loss=0.502738
  Epoch 1/3 | Batch 1800/3125 | loss=0.493137
  Epoch 1/3 | Batch 1900/3125 | loss=1.236349
  Epoch 1/3 | Batch 2000/3125 | loss=0.642294
  Epoch 1/3 | Batch 2100/3125 | lo

## Memory cleanup before distillation

In [9]:

import gc

# Free objects that are no longer needed before distillation.
# Metrics dicts have already been stored, so the model objects can be released.
for _var in ("ultra_teacher", "ultra_student", "student_seed_cls_model", "mg_cls",
             "pruned_no_kd_model", "pruned_finetuned_model", "ft_optimizer"):
    if _var in globals():
        del globals()[_var]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e6:.1f} MB allocated")
print("Memory cleanup done.")


GPU memory after cleanup: 507.8 MB allocated
Memory cleanup done.


## Distill teacher into pruned student on CIFAR10 images + labels

In [ ]:
kd_config_cls = DistillationLossConfig(
    alpha=KD_ALPHA,
    temperature=KD_TEMPERATURE,
)

optimizer = torch.optim.Adam(student_cls_model.parameters(), lr=lr)

distiller_cls = YOLOLogitsDistiller(
    teacher=teacher_cls_model,
    student=student_cls_model,
    kd_config=kd_config_cls,
    device=DEVICE,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_train_epochs=EPOCHS,
    eval_teacher=True,
)

loss_history, top1_history, top5_history = distiller_cls.train()


Epoch 1/3
  Batch 0001/3125 | total=2.323963 | hard=1.176126 | soft=3.089188
  Batch 0010/3125 | total=1.993177 | hard=1.264619 | soft=2.478883
  Batch 0020/3125 | total=1.696919 | hard=0.658140 | soft=2.389439
  Batch 0030/3125 | total=1.790586 | hard=0.974228 | soft=2.334824
  Batch 0040/3125 | total=1.905240 | hard=0.723141 | soft=2.693306
  Batch 0050/3125 | total=2.450083 | hard=1.581691 | soft=3.029011
  Batch 0060/3125 | total=1.805679 | hard=0.790644 | soft=2.482368
  Batch 0070/3125 | total=1.636666 | hard=0.631109 | soft=2.307038
  Batch 0080/3125 | total=2.091220 | hard=0.709779 | soft=3.012180
  Batch 0090/3125 | total=2.296005 | hard=1.038350 | soft=3.134442
  Batch 0100/3125 | total=2.412861 | hard=1.199761 | soft=3.221595
  Batch 0110/3125 | total=1.475563 | hard=0.343084 | soft=2.230549
  Batch 0120/3125 | total=1.635370 | hard=0.713756 | soft=2.249779
  Batch 0130/3125 | total=1.543295 | hard=0.735928 | soft=2.081540
  Batch 0140/3125 | total=2.126236 | hard=1.195496 |

## Evaluation: teacher vs pruned-only vs distilled student (CIFAR10 validation)

In [11]:
# --- Training loss summary ---
if "loss_history" in globals() and len(loss_history) > 0:
    print(f"Recorded {len(loss_history)} KD training batches")
    print(f"  First loss: {loss_history[0]:.6f}")
    print(f"  Last loss:  {loss_history[-1]:.6f}")
else:
    print("No training loss history found in current kernel session.")

# --- Evaluate teacher + distilled student via the distiller ---
print("\n--- CIFAR10 Validation Results ---\n")

eval_results = distiller_cls.evaluate()
teacher_metrics = eval_results.get("teacher")
student_metrics = eval_results["student"]
val_kd_loss = eval_results["val_kd_loss"]
kd_batches = eval_results["kd_batches"]

# --- Print results ---
if teacher_metrics is not None:
    print(f"Teacher (fine-tuned on CIFAR10):")
    print(
        f"  top1_acc={teacher_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={teacher_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={teacher_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={teacher_metrics['samples']}"
    )

if "pruned_no_kd_metrics" in globals() and pruned_no_kd_metrics is not None:
    print(f"\nPruned student (no finetuning, no KD, {PRUNE_SPARSITY*100:.0f}% sparsity):")
    print(
        f"  top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={pruned_no_kd_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={pruned_no_kd_metrics['samples']}"
    )

if "pruned_finetuned_metrics" in globals() and pruned_finetuned_metrics is not None:
    print(f"\nPruned student (finetuned {EPOCHS} epochs, no KD, {PRUNE_SPARSITY*100:.0f}% sparsity):")
    print(
        f"  top1_acc={pruned_finetuned_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={pruned_finetuned_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={pruned_finetuned_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={pruned_finetuned_metrics['samples']}"
    )

print(f"\nDistilled student (pruned + {EPOCHS} epochs, alpha={KD_ALPHA}, T={KD_TEMPERATURE}):")
print(
    f"  top1_acc={student_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={student_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={student_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={student_metrics['samples']}"
)

print(f"\nValidation KD loss (teacher vs distilled student, T={KD_TEMPERATURE}): "
      f"{val_kd_loss:.6f} over {kd_batches} batches")

# --- Summary table ---
print("\n--- Summary ---")
print(f"{'Model':<45} {'Top-1 Acc':>10} {'CE Loss':>10}")
print(f"{'─'*45} {'─'*10} {'─'*10}")
if teacher_metrics is not None:
    print(f"{'Teacher (fine-tuned)':<45} {teacher_metrics['top1_acc']*100:>9.2f}% {teacher_metrics['avg_ce_loss']:>10.4f}")
if "pruned_no_kd_metrics" in globals() and pruned_no_kd_metrics is not None:
    print(f"{'Pruned (no finetune, no KD)':<45} {pruned_no_kd_metrics['top1_acc']*100:>9.2f}% {pruned_no_kd_metrics['avg_ce_loss']:>10.4f}")
if "pruned_finetuned_metrics" in globals() and pruned_finetuned_metrics is not None:
    print(f"{'Pruned (finetuned, no KD)':<45} {pruned_finetuned_metrics['top1_acc']*100:>9.2f}% {pruned_finetuned_metrics['avg_ce_loss']:>10.4f}")
print(f"{'Distilled student (pruned + KD)':<45} {student_metrics['top1_acc']*100:>9.2f}% {student_metrics['avg_ce_loss']:>10.4f}")


Recorded 9375 KD training batches
  First loss: 2.323963
  Last loss:  2.433313

--- CIFAR10 Validation Results ---

Teacher (fine-tuned on CIFAR10):
  top1_acc=84.94% | CE_loss=1.6302 | fwd_ms/batch=13.54 | samples=10000

Pruned student (no finetuning, no KD, 50% sparsity):
  top1_acc=57.47% | CE_loss=1.4056 | fwd_ms/batch=31.97 | samples=10000

Pruned student (finetuned 3 epochs, no KD, 50% sparsity):
  top1_acc=76.01% | CE_loss=0.8143 | fwd_ms/batch=33.72 | samples=10000

Distilled student (pruned + 3 epochs, alpha=0.6, T=4.0):
  top1_acc=79.10% | CE_loss=0.6604 | fwd_ms/batch=26.15 | samples=10000

Validation KD loss (teacher vs distilled student, T=4.0): 6.191305 over 625 batches

--- Summary ---
Model                                          Top-1 Acc    CE Loss
───────────────────────────────────────────── ────────── ──────────
Teacher (fine-tuned)                              84.94%     1.6302
Pruned (no finetune, no KD)                       57.47%     1.4056
Pruned (finetuned